In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df_tx = spark.table("silver_byma.tipo_cambio") \
    .withColumnRenamed("simbolo_titulo", "simbolo")

df_prices = spark.table("silver_byma.precios_historicos")

In [0]:
df_analysis = df_tx.join(
    df_prices,
    on=["simbolo", "fecha"],
    how="left"
)

In [0]:
dim_instrumento = df_tx.select(
    "simbolo",
    "descripcion_titulo"
).dropDuplicates()

dim_instrumento = dim_instrumento.withColumn(
    "tipo_instrumento",
    when(col("descripcion_titulo").contains("Cedear"), "CEDEAR")
    .when(col("simbolo").rlike("^[A-Z]{2,4}[0-9]{2}"), "BONO")
    .otherwise("ACCION")
)

dim_instrumento = dim_instrumento.withColumn(
    "instrumento_sk",
    monotonically_increasing_id()
)

In [0]:
df_cliente = df_tx.groupBy("id_cliente").agg(
    count("*").alias("cantidad_ops")
)

dim_cliente = df_cliente.withColumn(
    "segmento_actividad",
    when(col("cantidad_ops") >= 100, "HIGH")
    .when(col("cantidad_ops") >= 20, "MEDIUM")
    .otherwise("LOW")
)

window_cli = Window.orderBy("id_cliente")

dim_cliente = dim_cliente.withColumn(
    "cliente_sk",
    monotonically_increasing_id()
)

In [0]:
dim_fecha = df_tx.select("fecha").dropDuplicates()

dim_fecha = dim_fecha.withColumn("anio", year("fecha")) \
    .withColumn("mes", month("fecha")) \
    .withColumn("trimestre", quarter("fecha")) \
    .withColumn("dia_semana", dayofweek("fecha")) \
    .withColumn("es_fin_de_semana",
        when(dayofweek("fecha").isin(1,7), 1).otherwise(0)
    )

# feriado bursátil = no hay precio
fechas_con_precio = df_prices.select("fecha").distinct()

dim_fecha = dim_fecha.join(
    fechas_con_precio.withColumn("tiene_precio", lit(1)),
    on="fecha",
    how="left"
)

dim_fecha = dim_fecha.withColumn(
    "es_feriado_bursatil",
    when(col("tiene_precio").isNull(), 1).otherwise(0)
).drop("tiene_precio")

window_fecha = Window.orderBy("fecha")

dim_fecha = dim_fecha.withColumn(
    "fecha_sk",
    monotonically_increasing_id()
)

In [0]:
dim_canal = df_tx.select(
    col("origen").alias("nombre_canal")
).dropDuplicates()

dim_canal = dim_canal.withColumn(
    "tipo_canal",
    when(col("nombre_canal").contains("App"), "DIGITAL")
    .when(col("nombre_canal").contains("Web"), "DIGITAL")
    .when(col("nombre_canal").contains("API"), "PROGRAMATICA")
    .otherwise("OTRO")
)

window_canal = Window.orderBy("nombre_canal")

dim_canal = dim_canal.withColumn(
    "canal_sk",
    monotonically_increasing_id()
)

In [0]:
df_fact = df_analysis \
    .withColumn("monto_total_usd", col("cantidad") * col("precio_operado_usd")) \
    .withColumn("desvio", col("precio_operado_usd") - col("precio_usd")) \
    .withColumn("desvio_pct",
        when(col("precio_usd").isNotNull(),
            (col("precio_operado_usd") - col("precio_usd")) / col("precio_usd")
        )
    )\
    .withColumn(
        "flag_compro_caro",
        when(
            (col("tipoTran") == "Compra") & (col("desvio") > 0),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "flag_vendio_barato",
        when(
            (col("tipoTran") == "Venta") & (col("desvio") < 0),
            1
        ).otherwise(0)
    )

In [0]:
df_fact = df_fact \
    .join(dim_instrumento.select("simbolo", "instrumento_sk"), "simbolo") \
    .join(dim_cliente.select("id_cliente", "cliente_sk"), "id_cliente") \
    .join(dim_fecha.select("fecha", "fecha_sk"), "fecha") \
    .join(dim_canal.withColumnRenamed("nombre_canal", "origen"),
    "origen",
    "left"
)

In [0]:
df_fact = df_fact.drop(
    "simbolo",
    "id_cliente",
    "fecha",
    "origen"
)

In [0]:
fact_operaciones = df_fact.select(
    "id_transaccion",
    "instrumento_sk",
    "cliente_sk",
    "fecha_sk",
    "canal_sk",
    col("tipoTran").alias("tipo_op"),
    "cantidad",
    "precio_operado_usd",
    "monto_total_usd",
    col("precio_usd").alias("precio_mercado_usd"),
    "desvio",
    "desvio_pct",
    "flag_compro_caro",
    "flag_vendio_barato"
)

fact_operaciones = fact_operaciones.orderBy("fecha_sk")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold_byma")

In [0]:
dim_instrumento.write.mode("overwrite").format("delta").saveAsTable("gold_byma.dim_instrumento")

dim_cliente.write.mode("overwrite").format("delta").saveAsTable("gold_byma.dim_cliente")

dim_fecha.write.mode("overwrite").format("delta").saveAsTable("gold_byma.dim_fecha")

dim_canal.write.mode("overwrite").format("delta").saveAsTable("gold_byma.dim_canal")

fact_operaciones.write.mode("overwrite").format("delta").saveAsTable("gold_byma.fact_operaciones")